In [1]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import os
import requests
import numpy
import pandas
import joblib

In [2]:
from sklearn.metrics import mean_absolute_error
from sklearn.svm import SVR
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import AdaBoostRegressor

In [3]:
train = pandas.read_csv('/kaggle/input/equity-post-HCT-survival-predictions/train.csv').drop(columns=['efs_time','ID'])
y_train = train['efs'].fillna(0)
for i in train.columns:
    train[i]=pandas.to_numeric(train[i], errors='coerce').fillna(0).astype(numpy.int64)
X_train = train.drop(columns=['efs'])
test = pandas.read_csv('/kaggle/input/equity-post-HCT-survival-predictions/test.csv').drop(columns=['ID']).fillna(0)
for i in test.columns:
    test[i]=pandas.to_numeric(test[i], errors='coerce').fillna(0).astype(numpy.int64)

X_train.head()

,dri_score,psych_disturb,cyto_score,diabetes,hla_match_c_high,hla_high_res_8,tbi_status,arrhythmia,hla_low_res_6,graft_type,...,karnofsky_score,hepatic_mild,tce_div_match,donor_related,melphalan_dose,hla_low_res_8,cardiac,hla_match_drb1_high,pulm_moderate,hla_low_res_10
0,0,0,0,0,0,0,0,0,6,0,...,90,0,0,0,0,8,0,2,0,10
1,0,0,0,0,2,8,0,0,6,0,...,90,0,0,0,0,8,0,2,0,10
2,0,0,0,0,2,8,0,0,6,0,...,90,0,0,0,0,8,0,2,0,10
3,0,0,0,0,2,8,0,0,6,0,...,90,0,0,0,0,8,0,2,0,10
4,0,0,0,0,2,8,0,0,6,0,...,90,0,0,0,0,8,0,2,0,10


In [4]:
test.columns, train.columns

(Index(['dri_score', 'psych_disturb', 'cyto_score', 'diabetes',
        'hla_match_c_high', 'hla_high_res_8', 'tbi_status', 'arrhythmia',
        'hla_low_res_6', 'graft_type', 'vent_hist', 'renal_issue',
        'pulm_severe', 'prim_disease_hct', 'hla_high_res_6', 'cmv_status',
        'hla_high_res_10', 'hla_match_dqb1_high', 'tce_imm_match', 'hla_nmdp_6',
        'hla_match_c_low', 'rituximab', 'hla_match_drb1_low',
        'hla_match_dqb1_low', 'prod_type', 'cyto_score_detail',
        'conditioning_intensity', 'ethnicity', 'year_hct', 'obesity', 'mrd_hct',
        'in_vivo_tcd', 'tce_match', 'hla_match_a_high', 'hepatic_severe',
        'donor_age', 'prior_tumor', 'hla_match_b_low', 'peptic_ulcer',
        'age_at_hct', 'hla_match_a_low', 'gvhd_proph', 'rheum_issue',
        'sex_match', 'hla_match_b_high', 'race_group', 'comorbidity_score',
        'karnofsky_score', 'hepatic_mild', 'tce_div_match', 'donor_related',
        'melphalan_dose', 'hla_low_res_8', 'cardiac', 'hla_match

In [5]:
svr = make_pipeline(StandardScaler(), SVR(kernel="rbf", C=1.0, epsilon=0.2))
model = AdaBoostRegressor(base_estimator=svr, learning_rate=1, n_estimators=100)
model.fit(X_train, y_train)

AdaBoostRegressor(base_estimator=Pipeline(steps=[('standardscaler',
                                                  StandardScaler()),
                                                 ('svr', SVR(epsilon=0.2))]),
                  learning_rate=1, n_estimators=100)

In [6]:
joblib.dump(model, 'model.pkl')

['model.pkl']

In [7]:
id = pandas.read_csv('/kaggle/input/equity-post-HCT-survival-predictions/test.csv')['ID']
test_predictions = model.predict(test)
test_predictions

array([0.40323996, 0.49214039, 0.4649512 ])

In [8]:
submission = pandas.DataFrame({
    'id': id.values,
    'sales_hat': test_predictions,
})
submission

,id,sales_hat
0,28800,0.403240
1,28801,0.492140
2,28802,0.464951


In [9]:
submission.to_csv('submission.csv', index = False)